# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dyajaballh8/FlyRank_Intern/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Distributions

I inspected the distributions of the main search-performance signals used in this project:

- GSC impressions
- GSC clicks
- Average search position
- CTR

The distributions are expected to be highly skewed. Most content items receive relatively low search volume, while a smaller number of items receive very high impressions and clicks.

Because of these heavy tails, raw averages alone can be misleading, so percentile and bucket-based checks are useful for interpreting the signals.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_auth (
    TYPE HTTP,
    BEARER_TOKEN '{HF_TOKEN}'
)
""")

DATA_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-05/data_0.parquet"
)

distribution_query = f"""
SELECT
    COUNT(*) AS rows,

    AVG(gsc_impressions) AS avg_impressions,
    MEDIAN(gsc_impressions) AS median_impressions,
    MAX(gsc_impressions) AS max_impressions,

    AVG(gsc_clicks) AS avg_clicks,
    MEDIAN(gsc_clicks) AS median_clicks,
    MAX(gsc_clicks) AS max_clicks,

    AVG(gsc_avg_position) AS avg_position,
    MEDIAN(gsc_avg_position) AS median_position,

    AVG(
        gsc_clicks * 100.0 /
        NULLIF(gsc_impressions, 0)
    ) AS avg_ctr

FROM read_parquet('{DATA_PATH}')

WHERE gsc_data_available = TRUE
"""

distribution_result = con.execute(
    distribution_query
).fetchdf()

display(distribution_result)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,avg_impressions,median_impressions,max_impressions,avg_clicks,median_clicks,max_clicks,avg_position,median_position,avg_ctr
0,4373422,60.859642,10.0,47607,0.211335,0.0,230,21.317578,11.903226,0.338005


## Signal tests

I tested three observable search-performance signals:

### Signal 1 — Search visibility

The assumption is that content with higher impressions has more potential value because more users are seeing it in search results.

**Verdict: CONFIRMED**

### Signal 2 — CTR opportunity

The assumption is that pages with high impressions and low CTR may have an opportunity for improvement.

**Verdict: CONFIRMED**

### Signal 3 — Search position

The assumption is that content appearing closer to the top of search results tends to receive stronger click-through performance.

**Verdict: CONFIRMED**

These tests are descriptive and directional. They support prioritization but do not prove causation.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal 1 — Search visibility

signal_1_query = f"""
SELECT
    CASE
        WHEN gsc_impressions >= 500 THEN 'HIGH'
        WHEN gsc_impressions >= 100 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS impressions_bucket,

    COUNT(*) AS n,

    AVG(gsc_clicks) AS avg_clicks

FROM read_parquet('{DATA_PATH}')

WHERE gsc_data_available = TRUE

GROUP BY 1

ORDER BY
    CASE impressions_bucket
        WHEN 'HIGH' THEN 1
        WHEN 'MEDIUM' THEN 2
        ELSE 3
    END
"""

signal_1 = con.execute(signal_1_query).fetchdf()

print("Signal 1 — Search visibility")
display(signal_1)


# Signal 2 — CTR opportunity

signal_2_query = f"""
SELECT
    CASE
        WHEN gsc_impressions >= 500
             AND gsc_clicks * 100.0 /
                 NULLIF(gsc_impressions, 0) < 0.5
        THEN 'HIGH_IMPRESSIONS_LOW_CTR'

        WHEN gsc_impressions >= 500
        THEN 'HIGH_IMPRESSIONS_OTHER_CTR'

        ELSE 'OTHER'
    END AS ctr_bucket,

    COUNT(*) AS n,

    AVG(
        gsc_clicks * 100.0 /
        NULLIF(gsc_impressions, 0)
    ) AS avg_ctr

FROM read_parquet('{DATA_PATH}')

WHERE gsc_data_available = TRUE
  AND gsc_impressions > 0

GROUP BY 1

ORDER BY n DESC
"""

signal_2 = con.execute(signal_2_query).fetchdf()

print("\nSignal 2 — CTR opportunity")
display(signal_2)


# Signal 3 — Search position

signal_3_query = f"""
SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN 'TOP_3'
        WHEN gsc_avg_position <= 10 THEN 'TOP_10'
        WHEN gsc_avg_position <= 20 THEN 'TOP_20'
        ELSE 'BELOW_20'
    END AS position_bucket,

    COUNT(*) AS n,

    AVG(
        gsc_clicks * 100.0 /
        NULLIF(gsc_impressions, 0)
    ) AS avg_ctr

FROM read_parquet('{DATA_PATH}')

WHERE gsc_data_available = TRUE
  AND gsc_impressions > 0
  AND gsc_avg_position > 0

GROUP BY 1

ORDER BY
    CASE position_bucket
        WHEN 'TOP_3' THEN 1
        WHEN 'TOP_10' THEN 2
        WHEN 'TOP_20' THEN 3
        ELSE 4
    END
"""

signal_3 = con.execute(signal_3_query).fetchdf()

print("\nSignal 3 — Search position")
display(signal_3)

print("\nVerdicts:")
print("Signal 1: CONFIRMED")
print("Signal 2: CONFIRMED")
print("Signal 3: CONFIRMED")


Signal 1 — Search visibility


,impressions_bucket,n,avg_clicks
0,HIGH,97611,3.429070
1,MEDIUM,455216,0.778255
2,LOW,3820595,0.061579


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Signal 2 — CTR opportunity


,ctr_bucket,n,avg_ctr
0,OTHER,4275811,0.338306
1,HIGH_IMPRESSIONS_LOW_CTR,76359,0.165996
2,HIGH_IMPRESSIONS_OTHER_CTR,21252,0.895524



Signal 3 — Search position


,position_bucket,n,avg_ctr
0,TOP_3,201212,0.866284
1,TOP_10,1598290,0.439150
2,TOP_20,871354,0.331308
3,BELOW_20,1566164,0.141595



Verdicts:
Signal 1: CONFIRMED
Signal 2: CONFIRMED
Signal 3: CONFIRMED


## Flag-linked test: CTR versus search position

I tested the assumption behind CTR-related prioritization.

The test focuses on pages with meaningful search visibility and compares CTR across search-position buckets.

The expectation is that pages with good search positions but unusually low CTR may represent an actionable opportunity.

**Verdict: CONFIRMED**

The observed data contains content items with strong visibility and positions within the first 20 results but very low CTR. These items are reasonable candidates for a CTR review, although the signal alone does not prove that changing a title or snippet will improve performance.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
flag_test_query = f"""
WITH base AS (

    SELECT
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,

        gsc_clicks * 100.0 /
        NULLIF(gsc_impressions, 0) AS ctr

    FROM read_parquet('{DATA_PATH}')

    WHERE gsc_data_available = TRUE
      AND gsc_impressions >= 500
      AND gsc_avg_position > 0
      AND gsc_avg_position <= 20
)

SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN 'TOP_3'
        WHEN gsc_avg_position <= 10 THEN 'TOP_10'
        ELSE 'TOP_20'
    END AS position_bucket,

    COUNT(*) AS n,

    ROUND(AVG(ctr), 3) AS avg_ctr,

    SUM(
        CASE
            WHEN ctr < 0.5 THEN 1
            ELSE 0
        END
    ) AS low_ctr_rows,

    ROUND(
        SUM(
            CASE
                WHEN ctr < 0.5 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS low_ctr_percentage

FROM base

GROUP BY 1

ORDER BY
    CASE position_bucket
        WHEN 'TOP_3' THEN 1
        WHEN 'TOP_10' THEN 2
        ELSE 3
    END
"""

flag_test = con.execute(
    flag_test_query
).fetchdf()

print("Flag-linked test — CTR vs search position")
display(flag_test)

print(
    "\nVerdict: CONFIRMED — "
    "high-visibility pages with good positions and low CTR exist in the data."
)

Flag-linked test — CTR vs search position


,position_bucket,n,avg_ctr,low_ctr_rows,low_ctr_percentage
0,TOP_3,4803,0.645,2293.0,47.74
1,TOP_10,75926,0.327,59360.0,78.18
2,TOP_20,7584,0.313,6009.0,79.23



Verdict: CONFIRMED — high-visibility pages with good positions and low CTR exist in the data.


## What this means in practice

The signal audit suggests that search visibility, search position, and CTR can be used as decision-support signals for prioritizing content review.

Content with high impressions and a strong search position but unusually low CTR should be investigated first because it already has measurable visibility. However, the signal should not be treated as proof that changing the page will improve performance.

Search intent, brand effects, query type, and recent changes may explain some low-CTR cases, so the recommendation is to review these pages rather than automatically modify them.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
summary_query = f"""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN gsc_impressions >= 500
            THEN 1
            ELSE 0
        END
    ) AS high_impression_rows,

    SUM(
        CASE
            WHEN gsc_impressions >= 500
             AND gsc_avg_position <= 20
             AND gsc_clicks * 100.0 /
                 NULLIF(gsc_impressions, 0) < 0.5
            THEN 1
            ELSE 0
        END
    ) AS ctr_review_candidates

FROM read_parquet('{DATA_PATH}')

WHERE gsc_data_available = TRUE
"""

summary = con.execute(summary_query).fetchdf()

print("Practical summary")
display(summary)

Practical summary


,total_rows,high_impression_rows,ctr_review_candidates
0,4373422,97611.0,67662.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.